# 32b — Cluster-level malignancy in blood: your decisions and manual calls

The blood analogue of `26b_cluster_malignancy`, on the same two jobs with `--tissue blood`.
Everything that differs is a property of the compartment, not a different method.

| | where | what |
|---|---|---|
| clustering | `jobs/run_cluster_malig.sh --tissue blood` | per **donor** Leiden 1.0 → UMAP; cluster pseudobulks; TRBV/TRAV call; ALICE family |
| Step 1 | `jobs/run_cluster_malig_model.sh --tissue blood` | cluster ALICE label; scores compared leave-one-study-out; cutoffs from held-out P |
| Step 2 | same job | HC → benign · ALICE donors keep ALICE · clear → auto · rest → review |
| **Decision 1** | here | which score |
| **Decision 2** | here | how strict (`MAX_ERR`) |
| **Manual calls** | `tables/blood_cluster_malig_review_v1.csv` | `m` / `b`, using `figures/nb32b_review/` |
| **Merge** | here | → `blood_T_malignancy_cluster_v1.parquet` → `malignancy.load_malignancy(..., tissue="blood")["mal_cluster"]` |

### Four things that are different in blood, and why

1. **The unit is the donor, not `sample_id`.** The geskin `sample_id` is an HTO lane pooling 5–6
   donors; a cluster spanning one lane would span several patients and its ALICE family share
   would mean nothing. The column is still called `sample_id` so the shared code is unchanged.
2. **The benign anchor is scarce.** SS blood is ~92% malignant CD4, so a benign cluster that is
   ≥80% CD4 barely exists; `MIN_CD4` is relaxed to 0.5 and the HC donors plus CD8 carry the
   negative side. If `consensus_signature` raises "no sample has both", that is this problem.
3. **The markers are the Sézary panel**, not the skin one. `CCR7`, `SELL` and `TCF7` are
   deliberately **not** in `CTCL_DOWN`: Sézary cells are central-memory-like, so their positivity
   is not evidence that a cluster is benign — the single most common way to mis-call SS blood.
4. **There is no `knn_frac` arm.** `mal_knn` is a skin-only call from the retired nb26, so that
   reference column is NaN here.

**Prerequisite:** `32_malignancy_tcr_cnv` rerun on the v2 annotation — `32b` reads
`alice_malignancy_blood_v2.parquet` and `tables/blood_tcr_clonality_per_sample.csv` (Step 3b).
The August `_v1` artifacts are from the v1 cohort and their cell ids do not match this object.

In [ ]:
from pathlib import Path


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir(); print("NB_DIR =", NB_DIR)
N_SHOW = 3          # review sheets shown inline (all are in figures/nb32b_review/)

In [ ]:
import sys, json, re, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

sys.path.insert(0, str(NB_DIR / "helpers"))
import cluster_malig_helpers as CM
import malignancy as MAL
for _m in (CM, MAL):
    importlib.reload(_m)

# Point BOTH modules at blood before any path is read. Without this the notebook would load the
# skin parquets, reindex them onto blood cell ids and report an empty call as a real one.
CM.use_tissue("blood")
MAL.use_tissue("blood")
print("tissue:", CM.TISSUE, "| object:", CM.TCR_OBJ.name, "| counts layer:", CM.COUNTS)
print("group key:", CM.GROUP_KEY, "| cell types:", CM.CELL_TYPES)
print("ALICE:", CM.ALICE_PQ.name, f"({CM.ALICE_COL})")
print("markers UP :", CM.CTCL_UP)
print("markers DOWN:", CM.CTCL_DOWN, " <- note CCR7/SELL/TCF7 are deliberately absent")

## What the jobs produced

Sample roles are as in skin: `train` (clonal ALICE donors), `train_hc` (healthy controls, the
negatives), `tcr_nonclonal` (ALICE donors with no founder family — their labels are kept but not
trained on), `apply` (everything else) and `excluded` (γδ / CD8-aggressive entities).

The purity ceiling is the best a cluster-level call can do at res 1.0: the share of TCR+ cells a
call gets right if every cluster took its majority ALICE label.

In [ ]:
man = json.loads(CM.MANIFEST_JSON.read_text())
res = json.loads(CM.MODEL_JSON.read_text())
cl = pd.read_csv(CM.CLUSTERS_CSV, index_col="cluster")

print(f"clustering: {man['n_samples']} {man.get('group_key', 'sample')}s / "
      f"{man['n_clusters']:,} clusters / {man['n_cells']:,} cells | res {man['params']['res']} "
      f"| written {man['written']}")
print(f"  skipped (<= 30 T cells): {len(man['skipped'])} | failed: {man['failed']}")
print(f"model run: {res['written']} | score = {res['program']} (asked: {res['program_arg']}) | "
      f"MAX_ERR {res['max_err']:.0%} | cutoffs {res['p_lo']:.3f} / {res['p_hi']:.3f}")
print(f"  TCR+ in mixed clusters {res['mixed_tcr_frac']:.1%} | purity ceiling "
      f"{res['purity_ceiling']:.1%} | HC cells the score alone calls malignant "
      f"{res['hc_fp_cells']:.1%}")

print("\nroles:")
print(cl.groupby("role").agg(samples=("sample_id", "nunique"), clusters=("n_cells", "size"),
                             cells=("n_cells", "sum")).to_string())

# The blood-specific risk, as a number: how much benign CD4 is there to anchor the program on?
if "frac_CD4" in cl:
    ben = cl[cl["tcr_label"].eq("benign")]
    print(f"\nbenign-labelled clusters: {len(ben)} | of those >= 50% CD4: "
          f"{int(ben['frac_CD4'].ge(0.5).sum())} | >= 80% CD4 (the skin gate): "
          f"{int(ben['frac_CD4'].ge(0.8).sum())}")
    print("If the >=80% count is near zero, MIN_CD4=0.5 is what made this run possible at all.")

## Decision 1 — which score?

Each candidate is scored leave-one-study-out on the ALICE-labelled clusters; the program is
relearned inside every fold.

- **program** — genes moving the same way in ≥75% of donors, malignant vs benign clusters within
  the same donor.
- **ctcl_apriori** — the Sézary marker lists above.
- **vskew** — how dominant the cluster's top TRBV/TRAV is against a polyclonal null.
- **program+vskew** — both.
- **knn_frac** — NaN in blood (no `mal_knn`); the column is kept so the tables keep their shape.

In blood, expect **vskew to carry more weight than in skin**: SS clones are large and the blood
studies are 5′ with real V reads, where several skin cohorts have none.

In [ ]:
cv = pd.read_csv(CM.CV_CSV)
cols = ["coverage", "auc_cluster", "auc_cells", "auc_within_median", "auto_cells_frac",
        "err_mal", "err_ben", "p_lo", "p_hi"]
print(f"held-out, pooled (MAX_ERR {res['max_err']:.0%}):")
print(cv[cv["fold"].eq("pooled")].set_index("candidate")[cols].round(3).to_string())
print("\nauto_cells_frac within each held-out study:")
print(cv[cv["fold"].ne("pooled")].pivot(index="fold", columns="candidate",
                                        values="auto_cells_frac").round(2).to_string())
_fig = CM.FIG_DIR / "nb32b_candidates_auto_share.png"
if _fig.exists():
    display(Image(str(_fig), width=500))

In [ ]:
prog = pd.read_csv(CM.PROGRAM_CSV)
n_folds = len(res["folds"])
print(f"learned program: {int(prog.direction.eq('up').sum())} up / "
      f"{int(prog.direction.eq('down').sum())} down, from "
      f"{res['n_train_donors_program']} donors")
for d in ("up", "down"):
    g = prog[prog["direction"].eq(d)]
    print(f"\n{d}:")
    print(", ".join(f"{r.gene}({r.n_folds_selected})" for r in g.itertuples()))

# Sanity: the program must not be a memory/naive axis. In SS that is the classic confound --
# Sezary cells are central-memory-like, so a program that separates CM from EM separates the
# arms for the wrong reason.
MEMORY = {"CCR7", "SELL", "TCF7", "LEF1", "IL7R", "S1PR1", "KLF2", "GZMB", "GNLY", "NKG7",
          "CX3CR1", "KLRG1"}
hits = sorted(set(prog["gene"]) & MEMORY)
print(f"\nmemory/differentiation genes in the learned program: {hits or 'none'}")
if hits:
    print("*** Check these. A program resting on them is a CM-vs-EM axis, not a malignancy "
          "axis, and it will call every central-memory T cell in a healthy donor malignant. ***")

## Decision 2 — how strict?

`MAX_ERR` is the largest share of auto-called cells allowed to be wrong on the held-out ALICE
clusters, per side. Stricter means a longer review queue.

The clone-V check is on by default: an auto-benign cluster goes to the queue when it carries the
donor's clone V gene at ≥30% of its V calls. In blood this matters more than in skin — the
no-V(D)J cohorts are where a clonal cluster can still score low, and in SS a missed malignant
cluster is a large fraction of the compartment.

In [ ]:
queue = pd.read_csv(CM.QUEUE_CSV)
print(queue.round(3).to_string(index=False))
print(f"\ncurrent run: {res['program']} at MAX_ERR {res['max_err']:.0%}, clone-V "
      f"{'on' if res.get('clone_v_check', True) else 'off'} "
      f"({res['clone_v_flipped_to_review']} added) -> {res['review_open_clusters']} open clusters "
      f"in {res['review_open_samples']} donors")
print(cl[cl["source"].eq("review")].groupby("reason")["n_cells"]
      .agg(clusters="size", cells="sum").to_string())
print("\nto change:  bash jobs/run_cluster_malig_model.sh --tissue blood --program <score> "
      "--max-err <x> [--no-clone-v]")

## Manual calls

Sheets are in `figures/nb32b_review/`. Fill `user_call` in
`tables/blood_cluster_malig_review_v1.csv` — `malignant`/`m`, `benign`/`b`, blank or `unsure`
stays undefined (`review_pending`) — then run **Merge**. Re-running the model job keeps filled rows.

**Reading a blood sheet.** Two traps specific to this compartment:

- A cluster that is CD7-low / CD26-low **and** carries the donor's clone V is malignant even when
  the program score is middling: in SS those two losses are the diagnostic immunophenotype.
- A CCR7+ SELL+ TCF7+ cluster is **not** benign by that fact alone. Sézary cells are
  central-memory-like; this is the single most common way to mis-call SS blood.

Whole donors can be dropped via `tables/blood_cluster_malig_excluded_samples_v1.csv`
(`sample_id`, `note`), which beats every other call.

In [ ]:
rev = pd.read_csv(CM.REVIEW_CSV, index_col="cluster",
                  dtype={"user_call": str, "note": str}, keep_default_na=False)
open_ = rev[rev["user_call"].str.strip().eq("")]
print(f"review queue: {len(rev)} rows | open {len(open_)} clusters / "
      f"{open_['n_cells'].astype(int).sum():,} cells in {open_['sample_id'].nunique()} donors "
      f"| filled {len(rev) - len(open_)}")
if len(open_):
    print(open_.groupby("study").agg(donors=("sample_id", "nunique"),
                                     clusters=("n_cells", "size"),
                                     cells=("n_cells", "sum")).to_string())
print(f"\nPDF: {CM.REVIEW_DIR / '_all_review_sheets.pdf'}")
for sid in list(dict.fromkeys(open_["sample_id"]))[:N_SHOW]:
    png = CM.REVIEW_DIR / (re.sub(r"[^A-Za-z0-9_.-]", "_", sid) + ".png")
    if png.exists():
        display(Image(str(png), width=1100))

## Cell-type QC — review clusters that are not what `nb31` called them

The 27-cluster queue surfaced an annotation problem rather than a malignancy one. Every open
cluster was called **benign** (2026-10-24), and reading them against the cluster pseudobulk sorts
them into three kinds:

- **`CD8A`+ `CD8B`− cytotoxic clusters with almost no TCR.** NK cells carry CD8 as an αα
  homodimer and so are `CD8A`+ `CD8B`−; a CD8 T cell is `CD8B`+. `nb31` labelled on `CD8A`, so NK
  clusters arrived here as CD8. `CM.nk_vs_t_scores` separates them on two marker sets, and
  `n_tcr / n_cells` is the independent check — an NK cluster has no rearranged TRB, so a cluster
  that scores NK *and* recovers no TCR is NK twice over. **These are relabelled `NK`.**
- **Clusters already labelled `NK`** (`frac_NK ≈ 1`, `n_tcr = 0`). No change: being NK is exactly
  what takes them out of the CD4 malignancy question.
- **Genuine CD8 T** (`CD8B`+, TCR recovered) and the CD4/mixed clusters. Left alone.

Writes `data/atlas_joint/blood_T_celltype_corrections_v1.csv`, which `CM.read_obs` overlays on
`cell_type_T` for every downstream consumer (the target screen reads it through that function).
`blood_T_annotated.h5ad` is **not** rewritten: the object stays the record of what `nb31` called,
this file is the record of what the review found, and `cell_type_T_nb31` is kept beside the
corrected column so the difference is always visible.

In [ ]:
sc = CM.nk_vs_t_scores()
sc["nb31"] = np.select([sc["frac_NK"] > 0.5, sc["frac_CD8"] > 0.5, sc["frac_CD4"] > 0.5],
                       ["NK", "CD8", "CD4"], default="mixed")
cols = ["n_cells", "n_tcr", "frac_CD4", "frac_CD8", "frac_NK", "t_score", "nk_score", "delta",
        "tcr_rate", "nb31", "nk_like"]
print(f"rule: nk_score - t_score >= {CM.NK_DELTA} AND tcr_rate < {CM.NK_MAX_TCR}\n")
print(sc.loc[rev.index, cols].sort_values("delta", ascending=False).round(2).to_string())

fix = sc.loc[rev.index].query("nb31 == 'CD8' and nk_like")
cells_cl = pd.read_parquet(CM.CLUSTERS_PQ, columns=["cell_id", "cluster"])
corr = (cells_cl[cells_cl["cluster"].isin(fix.index)]
        .assign(cell_type_T="NK", cell_type_T_nb31="CD8",
                reason=f"nb32b review: nk-t delta >= {CM.NK_DELTA}, tcr_rate < {CM.NK_MAX_TCR}"))
corr.to_csv(CM.CELLTYPE_FIX_CSV, index=False)
CM.CELLTYPE_FIX_CSV.with_suffix(".json").write_text(json.dumps(
    {"written": pd.Timestamp.now().isoformat(timespec="seconds"), "source": "notebooks/30_blood/32b",
     "rule": {"t_genes": CM.T_SCORE_GENES, "nk_genes": CM.NK_SCORE_GENES,
              "nk_delta": CM.NK_DELTA, "nk_max_tcr": CM.NK_MAX_TCR},
     "scope": "the nb32b review queue only", "clusters": sorted(fix.index),
     "n_cells": int(len(corr)), "from": "CD8", "to": "NK"}, indent=1))
print(f"\nCD8 -> NK: {len(fix)} clusters, {len(corr):,} cells -> {CM.CELLTYPE_FIX_CSV.name}")

# Atlas-wide context. NOT applied: the rule is calibrated on clusters that were looked at, and
# blood `cell_type_T` has only 606 NK cells in 667,584, which is not a credible NK fraction for
# PBMC. This is a finding about nb31, to be fixed there, not silently here.
wide = sc.query("nb31 == 'CD8' and nk_like")
cd8 = sc.query("nb31 == 'CD8'")
print(f"\nNOT APPLIED — atlas-wide, {len(wide)} of {len(cd8)} CD8-labelled clusters carry the "
      f"same signature ({int(wide['n_cells'].sum()):,} of {int(cd8['n_cells'].sum()):,} cells, "
      f"{wide['n_cells'].sum() / cd8['n_cells'].sum():.0%}). Blood CD8-vs-NK needs its own pass "
      "in nb31; until then the CD8 arm of the target screen is a CD8+NK mixture and the NK arm "
      "is too small to trust.")

## Merge

Applies every filled `user_call` on top of the job's calls and writes the per-cell
`blood_T_malignancy_cluster_v1.parquet`, which `malignancy.load_malignancy(..., tissue="blood")`
reads as `mal_cluster`. The base table is never overwritten, so this cell is re-runnable.

In [ ]:
cells_pq = pd.read_parquet(CM.CLUSTERS_PQ, columns=["cell_id", "sample_id", "cluster"])
out, final = CM.finalize_calls(
    cl[["sample_id", "study", "n_cells", "source", "call", "prob"]], cells_pq, CM.REVIEW_CSV)
out.to_parquet(CM.CALLS_PQ, index=False)
print(final.groupby("source")["n_cells"].agg(clusters="size", cells="sum").to_string())
print(f"\nwrote {CM.CALLS_PQ.name}: {len(out):,} cells | mal_cluster defined on "
      f"{out['mal_cluster'].notna().mean():.1%} | malignant where defined "
      f"{out['mal_cluster'].mean():.1%}")

In [ ]:
# ---- what the call says, and how it sits next to the TCR and CNV arms ----
smp = (out.assign(study=out["sample_id"].map(final.drop_duplicates("sample_id")
                                             .set_index("sample_id")["study"]),
                  m=out["mal_cluster"].astype("Float64"))
       .groupby(["study", "sample_id"])["m"].mean().dropna().reset_index())
studies = smp.groupby("study")["m"].median().sort_values().index
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(5, 3))
for i, s in enumerate(studies):
    v = smp.loc[smp["study"].eq(s), "m"]
    ax.scatter(i + rng.uniform(-0.2, 0.2, len(v)), v, s=8)
ax.set_xticks(range(len(studies)), studies, rotation=60, ha="right", fontsize=7)
ax.set_ylabel("malignant share of T cells")
ax.set_title(f"median donor is {smp['m'].median():.0%} malignant T cells "
             f"({len(smp)} donors)", fontsize=9)
fig.tight_layout()
fig.savefig(CM.FIG_DIR / "nb32b_malignant_share_by_study.png", dpi=150)
plt.show()

mal = MAL.load_malignancy(out["cell_id"], strict=False)
print(MAL.coverage_report(mal).to_string())
print()
print(MAL.concordance(mal).query("a == 'mal_cluster' or b == 'mal_cluster'").to_string())
print("\nHC donors should be ~0% malignant here BY CONSTRUCTION (the job assigns HC benign). "
      "That is an input, not a validation -- never quote it as evidence of specificity.")

## Next

`notebooks/40_targets/43_target_screen_blood` consumes this call. Build its input with

```
jobs/run_target_screen.sh --tissue blood
```